# Documentação: 08_ingestao_silver_all (Unificado)

Documentação técnica do notebook consolidado de Ingestão Silver (missões 03, 04, 05 e 07 do estágio).

> **Nota:** Os notebooks individuais 08, 09, 10 e 11 foram unificados neste notebook e depois deletados.

## Visão geral

| Missão | Tabela | Regra | Resultado (1ª execução) |
| --- | --- | --- | --- |
| 03 | ecommerce_produtos | preco_lista > 0 e < 5000 | 2832 Silver, 42 quarentena |
| 04 | ecommerce_categorias | Alertar se nº de categorias raiz mudar | 135 Silver, 12 raízes registradas |
| 05 | ecommerce_rastreamento_entregas | status_entrega na lista de valores válidos | 25 Silver, 0 quarentena |
| 07 | ecommerce_enderecos | estado deve ser UF válida (2 letras) | 14 Silver, 0 quarentena |

## Estrutura do notebook unificado (12 células)

| Célula | Tipo | Função |
| --- | --- | --- |
| 1 | Markdown | Título e visão geral das 4 missões |
| 2 | Python | `%run ./00_setup_config` — carrega credenciais .env |
| 3 | Python | Imports + config ADLS/UC + função `tabela_existe()` + cria tabela `controle_categorias_raiz` |
| 4 | Markdown | Separador — Missão 03 (produtos) |
| 5 | Python | **Missão 03:** Lê produtos, valida preco_lista > 0 e < 5000, grava Silver/Quarentena |
| 6 | Markdown | Separador — Missão 04 (categorias) |
| 7 | Python | **Missão 04:** Lê categorias, conta raízes, compara com lote anterior, alerta se mudou, grava Silver + controle |
| 8 | Markdown | Separador — Missão 05 (rastreamento) |
| 9 | Python | **Missão 05:** Lê rastreamento, valida status_entrega, grava Silver/Quarentena |
| 10 | Markdown | Separador — Missão 07 (enderecos) |
| 11 | Python | **Missão 07:** Lê enderecos, valida UF, grava Silver/Quarentena |
| 12 | Python | Resumo geral com totais de todas as missões |

## Arquitetura

Todos compartilham a mesma configuração (célula 3):
```
00_setup_config (credenciais .env)
    ↓
Teste de conectividade ADLS → se falhar, usa UC/Lakehouse Federation
    ↓
Para cada missão:
  Ler origem (sqlserver_catalog.squad2.*)
  → Filtro incremental (anti-join com Silver existente)
  → Aplicar regra de validação
  → Válidos → Silver (Delta, append)
  → Inválidos → Quarentena (Delta, append)
    ↓
Resumo geral (célula 12)
```

## Tabelas UC criadas

| Tabela | Função |
| --- | --- |
| `workspace.default.silver_ecommerce_produtos` | Silver de produtos (2832 registros) |
| `workspace.default.quarentena_ecommerce_produtos` | Produtos rejeitados (preço fora do range) |
| `workspace.default.silver_ecommerce_categorias` | Silver de categorias (135 registros) |
| `workspace.default.controle_categorias_raiz` | Controle de categorias raiz por lote |
| `workspace.default.silver_ecommerce_rastreamento` | Silver de rastreamento (25 registros) |
| `workspace.default.quarentena_ecommerce_rastreamento` | Rastreamento rejeitado (status inválido) |
| `workspace.default.silver_ecommerce_enderecos` | Silver de enderecos (14 registros) |
| `workspace.default.quarentena_ecommerce_enderecos` | Enderecos rejeitados (UF inválida) |

---
# Missão 03 — ecommerce_produtos (preco_lista)

## Regra
`preco_lista` deve ser > 0 e < 5000. Produtos fora do range vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_produtos` — 2874 linhas

## Colunas
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| sku | string | Identificador do produto |
| nome_produto | string | Nome do produto |
| descricao | string | Descrição |
| id_categoria | long | FK para ecommerce_categorias |
| preco_lista | double | Preço (validado) |
| unidade_medida | string | kg, L, etc. |
| nome_marca | string | Marca |
| is_ativo | boolean | Produto ativo? |

## Validação
- `PRECO_NULO` — preco_lista é NULL
- `PRECO_MENOR_OU_IGUAL_ZERO` — preco_lista ≤ 0
- `PRECO_MAIOR_OU_IGUAL_5000` — preco_lista ≥ 5000

## Resultado
- 2832 válidos → `workspace.default.silver_ecommerce_produtos`
- 42 rejeitados → `workspace.default.quarentena_ecommerce_produtos`
  - 41 com PRECO_MENOR_OU_IGUAL_ZERO
  - 1 com PRECO_MAIOR_OU_IGUAL_5000
- preco_lista min: -48.84, max: 7048.0

---
# 09_ingestao_silver_categorias — Missão 04

## Objetivo
Alertar se o número total de categorias raiz (`id_categoria_pai IS NULL`) mudar entre lotes. Não bloqueia o processamento — apenas emite alerta.

## Origem
`sqlserver_catalog.squad2.ecommerce_categorias` — 135 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_categoria | long | ID da categoria |
| nome_categoria | string | Nome |
| id_categoria_pai | double | ID da categoria pai (NULL = raiz) |
| nome_categoria_pai | string | Nome da categoria pai |
| tipo_categoria | string | "Categoria" (raiz) ou "Subcategoria" |

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback. Define tabela de controle `controle_categorias_raiz` |
| 5 | Criar tabela controle | Cria `workspace.default.controle_categorias_raiz` (total, nomes, timestamp) |
| 6 | Ler categorias origem | Lê `ecommerce_categorias` via Lakehouse Federation. Anti-join por `id_categoria` |
| 7 | Validar categorias raiz | Conta categorias com `id_categoria_pai IS NULL`. Compara com lote anterior (tabela de controle). Se diferente → alerta com detalhes (adicionadas/removidas) |
| 8 | Gravar Silver e controle | Append categorias na Silver. INSERT na tabela de controle com total atual |
| 9 | Resumo | Imprime totais, status do alerta e histórico de controle |

## Tabela de controle
```sql
CREATE TABLE workspace.default.controle_categorias_raiz (
    total_categorias_raiz  BIGINT,
    nomes_categorias_raiz STRING,
    dt_processamento       TIMESTAMP
) USING DELTA
```

## Resultado da execução
- 135 categorias na origem (todas novas — primeira carga)
- 12 categorias raiz identificadas:
  - Adega & Destilados, Bazar & Utilidades, Bebidas, Higiene & Cuidados Pessoais, Hortifruti Seco & Castanhas, Limpeza, Matinais Doces & Biscoitos, Mercearia Básica, Mercearia Salgada, Mundo Bebê, Padaria Industrializada, Pets
- 135 categorias gravadas na Silver
- Controle registrado: 12 raízes
- Sem alerta (primeira execução)

---
# 10_ingestao_silver_rastreamento — Missão 05

## Objetivo
Validar que `status_entrega` está na lista de valores do fluxo logístico. Registros com status fora do fluxo vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_rastreamento_entregas` — 25 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_rastreamento | long | ID do rastreamento |
| id_pedido_ecommerce | long | FK para ecommerce_pedidos |
| codigo_rastreio | string | Código de rastreio |
| id_transportadora | long | ID da transportadora |
| nome_transportadora | string | Nome da transportadora |
| status_entrega | string | Status (validado) |
| dt_evento | timestamp | Data do evento |
| observacao | string | Observação |
| bronze_source_file | string | Arquivo de origem |

## Lista de status válidos (confirmar com o professor)
```
aguardando separacao, em separacao, separado, em transito, entregue, falha, devolvido
+ variações com underscore: aguardando_separacao, em_separacao, em_transito
```

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback |
| 5 | Parametros status validos | Define `STATUS_VALIDOS` (lista de 10 valores) e função `tabela_existe()` |
| 6 | Ler rastreamento origem | Lê `ecommerce_rastreamento_entregas` via Lakehouse Federation. Anti-join por `id_rastreamento` |
| 7 | Validar status_entrega | Normaliza status (trim + lowercase). Classifica: `STATUS_NULO`, `STATUS_FORA_DO_FLOW` |
| 8 | Montar Silver e quarentena | Adiciona `silver_processed_at`. Separa válidos de inválidos |
| 9 | Gravar Silver e quarentena | Append em `silver_ecommerce_rastreamento` e `quarentena_ecommerce_rastreamento` |
| 10 | Resumo | Imprime totais, distribuição de quarentena e amostra da Silver |

## Resultado da execução
- 25 rastreamentos na origem (todos novos — primeira carga)
- 25 válidos → `workspace.default.silver_ecommerce_rastreamento`
- 0 rejeitados → todos com status "em separacao" (válido)
- Transportadoras: Correios, JadLog, Loggi, Sequoia

---
# 11_ingestao_silver_enderecos — Missão 07

## Objetivo
Validar que `estado` é uma UF válida (2 letras maiúsculas, correspondente a um estado brasileiro). Enderecos com UF inválida vão para quarentena.

## Origem
`sqlserver_catalog.squad2.ecommerce_enderecos` — 14 linhas

## Colunas da tabela
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| id_endereco | long | ID do endereço |
| id_cliente | long | FK para ecommerce_clientes |
| logradouro | string | Rua/avenida |
| numero | long | Número |
| complemento | string | Complemento |
| bairro | string | Bairro |
| cep | string | CEP |
| cidade | string | Cidade |
| estado | string | UF (validado) |
| latitude | double | Latitude |
| longitude | double | Longitude |
| apelido | string | Apelido (Casa, Trabalho) |
| is_principal | boolean | Endereço principal? |
| bronze_source_file | string | Arquivo de origem |

## Lista de UFs válidas (26 estados + DF)
```
AC, AP, AM, BA, CE, DF, ES, GO, MA, MT, MS, MG, PA, PB, PR, PE,
PI, RJ, RN, RS, RO, RR, SC, SP, SE, TO
```

## Células

| Célula | Título | Função |
| --- | --- | --- |
| 1 | Markdown | Título e descrição da regra |
| 2 | Setup | `%run ./00_setup_config` |
| 3 | Importacao | Importa bibliotecas |
| 4 | Configuracao camadas | ADLS + UC fallback |
| 5 | Parametros UF validas | Define `UFS_VALIDAS` (26 UFs) e função `tabela_existe()` |
| 6 | Ler enderecos origem | Lê `ecommerce_enderecos` via Lakehouse Federation. Anti-join por `id_endereco` |
| 7 | Validar UF | Normaliza estado (trim + uppercase). Classifica: `ESTADO_NULO`, `UF_FORMATO_INVALIDO`, `UF_NAO_RECONHECIDA` |
| 8 | Montar Silver e quarentena | Adiciona `silver_processed_at`. Separa válidos de inválidos |
| 9 | Gravar Silver e quarentena | Append em `silver_ecommerce_enderecos` e `quarentena_ecommerce_enderecos` |
| 10 | Resumo | Imprime totais, distribuição de quarentena e amostra da Silver |

## Resultado da execução
- 14 enderecos na origem (todos novos — primeira carga)
- 14 válidos → `workspace.default.silver_ecommerce_enderecos`
- 0 rejeitados → todos com UF válida
- Distribuição de UFs: SP=6, MG=4, ES=2, RJ=2

---
# Resumo geral — Status das 7 missões

| # | Tabela | Regra | Notebook | Status | Resultado |
| --- | --- | --- | --- | --- | --- |
| 01 | ecommerce_pedidos | Controle de hash (não reprocessar) | 06_ingestao_pedidos_bronze | ✅ Criado | Aguarda execução |
| 02 | ecommerce_itens_pedido | FK com Silver de pedidos + espera | 07_ingestao_silver_itens | ✅ Criado | Aguarda execução |
| 03 | ecommerce_produtos | preco_lista > 0 e < 5000 | 08_ingestao_silver_all (unificado) | ✅ Executado | 2832 Silver, 42 quarentena |
| 04 | ecommerce_categorias | Alerta de categorias raiz | 08_ingestao_silver_all (unificado) | ✅ Executado | 135 Silver, 12 raízes |
| 05 | ecommerce_rastreamento_entregas | status_entrega válido | 08_ingestao_silver_all (unificado) | ✅ Executado | 25 Silver, 0 quarentena |
| 06 | ecommerce_clientes | Email único no lote e Silver | 05_ingestao_silver (Kauan) | ✅ Executado | 361 Silver, 12 quarentena |
| 07 | ecommerce_enderecos | UF válida (2 letras) | 08_ingestao_silver_all (unificado) | ✅ Executado | 14 Silver, 0 quarentena |

## Pontos pendentes (conversar com Kauan)
1. **Schema de ecommerce_itens_pedido**: está em `squad1` (129 linhas) e `dbo` (258 linhas), não no `squad2`. Confirmar qual usar.
2. **Lista de status do fluxo logístico** (missão 05): confirmar com o professor se a lista está correta.
3. **ADLS**: notebooks 06 e 07 têm `raise NotImplementedError` no caminho ADLS. Kauan precisa preencher com a estrutura de pastas dele.
4. **Silver de pedidos** (missão 02): pré-requisito para a FK dos itens. Notebook 07 cria uma carga inicial simples, mas Kauan pode querer um notebook próprio com regras de qualidade.
5. **Ordem de execução recomendada**: 06 (Bronze pedidos) → 08_ingestao_silver_all (4 missões) → 05 (Silver clientes) → 07 (Silver itens com FK)